GAIA Data - Wow! Signal

The Wow! signal was a strong narrowband radio signal detected on August 15, 1977, by Ohio State University's Big Ear radio telescope. The signal appeared to show the expected hallmarks of extraterrestrial origin. The entire signal sequence lasted for the full 72-second window during which Big Ear was able to observe it, but has not been detected since.
A
recent study
suggests that the signal may have been due to a strongly magnetic star (magnetar).
The Wow! signal originated at the coordinates:
ra
= 291.79 deg,
dec
= -26.95 deg

In [ ]:
import numpy as np

from astropy.table import QTable
from astroquery.gaia import Gaia


Get a list of  the Gaia objects near the Wow! signal coordinates.

You want ALL objects with

ra
and
dec
within a circular radius of 1 degree of the Wow! signal coordinates
phot_g_mean_mag
< 14
parallax
> 1
bp_rp
IS NOT NULL
You also want the
source_id
for all objects
Make sure you adjust your
SELECT
to include all objects

In [ ]:
query_circle = """
SELECT TOP 1000
source_id, ra, dec, parallax, phot_g_mean_mag, bp_rp
FROM gaiadr3.gaia_source_lite
WHERE DISTANCE( POINT(291.79, -26.95), POINT(ra, dec) ) < 1.0 
AND phot_g_mean_mag < 14.0
AND parallax > 1.0
AND bp_rp IS NOT NULL
ORDER BY bp_rp ASC
"""


In [ ]:
my_job_query = Gaia.launch_job(query_circle)


In [ ]:
print(my_job_query)


In [ ]:
my_circle_table = my_job_query.get_results()


In [ ]:
my_circle_table


In [ ]:
my_circle_table.show_in_notebook()


Print the table row that contains the brightest object

In [ ]:
my_circle_table.sort('phot_g_mean_mag')


In [ ]:
my_circle_table[0]


Write a function to find the distance of an object

$$ \Large
\mathrm{Distance\ in\ pc}\ (d_{\mathrm{pc}}) = \frac{1}{\mathrm{p\,(mas)} / 1000 }
$$

In [ ]:
def my_distance(my_parallax):
    distance_in_pc=1/(my_parallax/1000)
    return distance_in_pc

distance=(my_distance(my_parallax=my_circle_table['parallax']))


Add a column
distance
to your table that has the distance (in parsecs) for all of the objects

In [ ]:
my_circle_table['distance']=distance


In [ ]:
my_circle_table


Write a function to find the Absolute G magnitude ($M_{G}$) of an object

$$ \Large
{\displaystyle M_{G} = m_{G} - 5\log _{10} \left( \frac{d_{\mathrm{pc}}} {10\, \mathrm{pc}} \right)}
$$
$m_{G}$ =
phot_g_mean_mag

In [ ]:
def abs_mag (my_distance, my_app_mag):
    my_abs_mag = my_app_mag - 5*(np.log10(my_distance/10))
    return my_abs_mag


Add a column
Abs G Mag
to your table that has the Absolute G magnitude for all of the objects

In [ ]:
Abs_G_Mag=(abs_mag(my_distance=my_circle_table['distance'], my_app_mag=my_circle_table['phot_g_mean_mag']))
my_circle_table['Abs_G_Mag']=Abs_G_Mag


In [ ]:
my_circle_table


Create a subset of the table with the following characteristics:

An Absolute G magnitude
brighter
than 4th magnitude
BP-RP
color index greater than 1.0
These are
Giant branch stars

You DO NOT need to a do a new Gaia search, just filter/mask the data from the table you already made.

The
Python_Tables
notebook may be helpful.

In [ ]:
my_giant_table=my_circle_table[(my_circle_table['bp_rp'] > 1.0)
    &(my_circle_table['Abs_G_Mag'] < 4.0)]
my_giant_table


What fraction of the total WOW! signal objects are Giant branch stars?

In [ ]:
np.size(my_giant_table) / np.size(my_circle_table)


Create a subset of the table with the following characteristics:

An Absolute G magnitude between 5th and 11th magnitude
BP-RP
color index between 1.1 and 2.2
These are
Main sequence K-type stars

In [ ]:
my_K_table=my_circle_table[(my_circle_table['bp_rp'] > 1.1) &
    (my_circle_table['bp_rp'] < 2.2) & (my_circle_table['Abs_G_Mag'] > 5) & (my_circle_table['Abs_G_Mag'] < 11)]


In [ ]:
my_K_table


What fraction of the total WOW! signal objects are Main sequence K-type stars?

In [ ]:
np.size(my_K_table) / np.size(my_circle_table)


Due Wed Jan 21 - 1 pm

Rerun your notebook:
Kernel -> Restart Kernal and Run All Cells
Save notebook as .html: File -> Download as -> HTML
Upload .html to Canvas